In [ ]:
#!pip install lightning

In [2]:
import pandas as pd
import os

# Load the CSV file
df = pd.read_csv("/kaggle/input/computer-vision-project-dataset/poster_image_scores.csv")

# Create 'poster_title' column:
# - os.path.basename(x): gets the file name from full path (e.g., 'Superman.jpg')
# - os.path.splitext(...)[0]: removes the file extension (e.g., 'Superman')
# - .replace("_", " "): replaces underscores with spaces (e.g., 'The_Conjuring_Last_Rites' → 'The Conjuring Last Rites')
df['poster_title'] = df['image_path'].apply(lambda x: os.path.splitext(os.path.basename(x))[0].replace('_', ' '))

# Preview the updated DataFrame
print(df[['image_path', 'poster_title']].head())

                                          image_path  \
0           poster_images/poster_images/Superman.jpg   
1  poster_images/poster_images/The_Conjuring_Last...   
2              poster_images/poster_images/Ligaw.jpg   
3  poster_images/poster_images/El_Cas_Àngelus_La_...   
4  poster_images/poster_images/Jurassic_World_Reb...   

                           poster_title  
0                              Superman  
1              The Conjuring Last Rites  
2                                 Ligaw  
3  El Cas Àngelus La fascinació de Dalí  
4                Jurassic World Rebirth  


In [3]:
face_matches_df = pd.read_csv("/kaggle/input/computer-vision-project-dataset/face_matches.csv")

In [4]:
df[df.image_path == "poster_images/poster_images/_Tis_the_Season_for_Love_2015_.jpg"]

,image_path,imdb_score,score_range,poster_title
29871,poster_images/poster_images/_Tis_the_Season_fo...,6.7,6–7,Tis the Season for Love 2015


In [5]:
df.to_csv("tesnime.csv",index=False)

In [6]:
df.shape

(49143, 4)

In [10]:
image_with_actors_df = pd.read_csv("/kaggle/input/computer-vision-project-dataset/poster_image_scores_with_actors.csv")

In [11]:
image_with_actors_df = image_with_actors_df[["image_path","popularity"]]


In [12]:
combined_df = pd.merge(
    image_with_actors_df,
    df,
    on='image_path',
    how='right'  # keeps only rows that are in imdb_df
)

# Check result
print(combined_df.head())
print(f"Combined row count: {len(combined_df)}")

                                          image_path  popularity  imdb_score  \
0           poster_images/poster_images/Superman.jpg        15.0         0.0   
1  poster_images/poster_images/The_Conjuring_Last...        15.0         0.0   
2              poster_images/poster_images/Ligaw.jpg        15.0         0.0   
3  poster_images/poster_images/El_Cas_Àngelus_La_...        15.0         0.0   
4  poster_images/poster_images/Jurassic_World_Reb...        15.0         0.0   

  score_range                          poster_title  
0         0–1                              Superman  
1         0–1              The Conjuring Last Rites  
2         0–1                                 Ligaw  
3         0–1  El Cas Àngelus La fascinació de Dalí  
4         0–1                Jurassic World Rebirth  
Combined row count: 49311


In [13]:
bert_embeddings = pd.read_csv("/kaggle/input/computer-vision-project-dataset/titles_with_bert_embeddings.csv")

In [14]:
bert_embeddings['bert_cls_embedding'] = bert_embeddings['bert_cls_embedding'].apply(
    lambda x: [float(i) for i in x.split(',')] if isinstance(x, str) else x
)

# Step 2 (Optional): Check result
print(len(bert_embeddings['bert_cls_embedding'].iloc[0]))# should show a list like [-0.86, -0.12, ...]
print(type(bert_embeddings['bert_cls_embedding'].iloc[0]))  # should be <class 'list'>

768
<class 'list'>


In [15]:
combined_df.shape

(49311, 5)

In [16]:
bert_embeddings= bert_embeddings[["image_path","bert_cls_embedding"]]

In [ ]:
final_df = pd.merge(
    combined_df,
    bert_embeddings,
    on='image_path',
    how='right'  # keep only those in bert_embeddings
)


print(final_df.head())
print(f"Final row count: {len(final_df)}")

                                          image_path  popularity  imdb_score  \
0           poster_images/poster_images/Superman.jpg        15.0         0.0   
1  poster_images/poster_images/The_Conjuring_Last...        15.0         0.0   
2              poster_images/poster_images/Ligaw.jpg        15.0         0.0   
3  poster_images/poster_images/El_Cas_Àngelus_La_...        15.0         0.0   
4  poster_images/poster_images/Jurassic_World_Reb...        15.0         0.0   

  score_range                          poster_title  \
0         0–1                              Superman   
1         0–1              The Conjuring Last Rites   
2         0–1                                 Ligaw   
3         0–1  El Cas Àngelus La fascinació de Dalí   
4         0–1                Jurassic World Rebirth   

                                  bert_cls_embedding  
0  [-0.86040306, -0.120517045, -0.83552206, 0.296...  
1  [-0.059585854, 0.2324819, -0.3084668, 0.053066...  
2  [-0.4294514, -0.1105

In [18]:
final_df.rename(columns={'popularity': 'actor_score'}, inplace=True)
final_df.rename(columns={'bert_cls_embedding': 'title_embedding'}, inplace=True)

In [ ]:
import os, random, time, gc
import pandas as pd
from PIL import Image, UnidentifiedImageError
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
import lightning as L
from torchmetrics.regression import MeanAbsoluteError, MeanSquaredError

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


POSTER_DIR = "/kaggle/input/computer-vision-project-dataset/poster_images/"
IMG_COL    = "image_path"
SCORE_COL  = "imdb_score"
EMBEDDING_DIM = 768
EPOCHS     = 10
BATCH_SIZE = 128
LR         = 1e-4


class PosterDS_Base(Dataset):
    def __init__(self, df, root, tfm):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.tfm = tfm

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.root, row[IMG_COL])

        try:
            img = Image.open(path).convert("RGB")
        except (FileNotFoundError, UnidentifiedImageError):
            return self.__getitem__((idx + 1) % len(self))

        x_img = self.tfm(img)
        y = torch.tensor(row[SCORE_COL], dtype=torch.float32)

        return x_img, y


class ResNet50Regressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet = timm.create_model("resnet50", pretrained=True)
        resnet_out_dim = self.resnet.get_classifier().in_features
        self.resnet.reset_classifier(0)

        self.head = nn.Sequential(
            nn.Linear(resnet_out_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 1)
        )

    def forward(self, x_img):
        x_resnet = self.resnet(x_img)
        return self.head(x_resnet).squeeze(1)


class Regr_Base(L.LightningModule):
    def __init__(self, model, lr=LR):
        super().__init__()
        self.model = model
        self.lr = lr
        self.mae = MeanAbsoluteError()
        self.mse = MeanSquaredError()

    def forward(self, x_img):
        return self.model(x_img)

    def _step(self, batch, tag):
        x_img, y = batch
        yhat = self(x_img)
        loss = nn.MSELoss()(yhat, y)
        self.log(f"{tag}_mae", self.mae(yhat, y), prog_bar=True, batch_size=len(y))
        self.log(f"{tag}_mse", self.mse(yhat, y), prog_bar=True, batch_size=len(y))
        return loss

    def training_step(self, b, i): return self._step(b, "train")
    def validation_step(self, b, i): return self._step(b, "val")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}


def build_loaders_base(df, img_size=224, bs=32, workers=2):
    df = df[df[IMG_COL].apply(lambda p: os.path.exists(os.path.join(POSTER_DIR, p)))]
    val_df = df.sample(frac=0.1, random_state=SEED)
    tr_df  = df.drop(val_df.index)

    mean, std = [0.5]*3, [0.5]*3
    tr_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(.2, .2, .2, .1),
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    val_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(), transforms.Normalize(mean, std)])

    tr_loader = DataLoader(
        PosterDS_Base(tr_df, POSTER_DIR, tr_tfm),
        batch_size=bs, shuffle=True, num_workers=workers, pin_memory=True)
    val_loader = DataLoader(
        PosterDS_Base(val_df, POSTER_DIR, val_tfm),
        batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
    return tr_loader, val_loader


def train_one_base(df, epochs=EPOCHS, bs=BATCH_SIZE, freeze_resnet=False):
    tr_loader, val_loader = build_loaders_base(df, bs=bs)
    model = ResNet50Regressor()

    if freeze_resnet:
        for param in model.resnet.parameters():
            param.requires_grad = False
        print("ResNet backbone is frozen.")
    else:
        print("ResNet backbone is trainable.")

    lit_model = Regr_Base(model)

    checkpoint_cb = L.pytorch.callbacks.ModelCheckpoint(
        monitor="val_mae", mode="min", save_top_k=1,
        filename="best-resnet50-{epoch:02d}-{val_mae:.3f}"
    )

    print(f"Training on {len(tr_loader.dataset)} samples")
    print(f"Validating on {len(val_loader.dataset)} samples")

    trainer = L.Trainer(
        max_epochs=epochs,
        precision="16-mixed" if torch.cuda.is_available() else 32,
        accelerator="auto",
        deterministic=True,
        log_every_n_steps=10,
        callbacks=[checkpoint_cb]
    )

    t0 = time.time()
    trainer.fit(lit_model, tr_loader, val_loader)
    metrics = trainer.validate(lit_model, val_loader, verbose=False)[0]
    print(f"\nRESNET50 MAE={metrics['val_mae']:.3f}  MSE={metrics['val_mse']:.3f}  Time={(time.time() - t0)/60:.1f} min")
    print(f"Best checkpoint saved to: {checkpoint_cb.best_model_path}")

    lit_model.eval()
    with torch.no_grad():
        for x_img, y_true in val_loader:
            y_pred = lit_model(x_img.to(lit_model.device))
            print("\nSample predictions:")
            for i in range(min(5, len(y_true))):
                print(f"True: {y_true[i].item():.2f} → Pred: {y_pred[i].item():.2f}")
            break

    return metrics


metrics_base = train_one_base(final_df)

print("\nFinal Base Model Metrics:")
print("MAE:", metrics_base["val_mae"], "MSE:", metrics_base["val_mse"])


INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs


ResNet backbone is trainable.
Training on 44380 samples
Validating on 4931 samples


2025-06-10 18:11:56.186217: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749579116.203070     122 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749579116.208121     122 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name  | Type              | Params | Mode 
----------------------------------------------------
0 | model | ResNet50Regressor | 24.6 M | train
1 | mae   | MeanAbsoluteError | 0      | train
2 | mse   | MeanSquaredError  | 0      | train
----------------------------------------------------
24.6 M    Trainable params
0         Non-trainable params
24.6 M    Total params
98.231 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]


RESNET50 MAE=1.854  MSE=6.005  Time=60.1 min
Best checkpoint saved to: /kaggle/working/lightning_logs/version_1/checkpoints/best-resnet50-epoch=02-val_mae=1.777.ckpt


In [ ]:

import os, random, time, gc
import pandas as pd
from PIL import Image, UnidentifiedImageError
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
import lightning as L
from torchmetrics.regression import MeanAbsoluteError, MeanSquaredError

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


POSTER_DIR = "/kaggle/input/computer-vision-project-dataset/poster_images/"
IMG_COL    = "image_path"
SCORE_COL  = "imdb_score"
EMBEDDING_DIM = 768
EPOCHS     = 10
BATCH_SIZE = 128
LR         = 1e-4

class PosterDS(Dataset):
    def __init__(self, df, root, tfm, embedding_dim):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.tfm = tfm
        self.embedding_dim = embedding_dim

        min_score = self.df["actor_score"].min()
        max_score = self.df["actor_score"].max()
        self.df["actor_score_norm"] = (self.df["actor_score"] - min_score) / (max_score - min_score)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.root, row[IMG_COL])

        try:
            img = Image.open(path).convert("RGB")
        except (FileNotFoundError, UnidentifiedImageError):
            return self.__getitem__((idx + 1) % len(self))

        x_img = self.tfm(img)
        y = torch.tensor(row[SCORE_COL], dtype=torch.float32)

        emb = torch.tensor(row["title_embedding"], dtype=torch.float32)

        if emb.size(0) > self.embedding_dim:
            emb = emb[:self.embedding_dim]
        elif emb.size(0) < self.embedding_dim:
            pad = torch.zeros(self.embedding_dim - emb.size(0))
            emb = torch.cat([emb, pad])

        actor_score = torch.tensor(row["actor_score_norm"], dtype=torch.float32).unsqueeze(0)
        return x_img, emb, actor_score, y


class ViTWithTabular(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.vit = timm.create_model("vit_base_patch16_224", pretrained=True)
        vit_out_dim = self.vit.head.in_features
        self.vit.head = nn.Identity()

        self.head = nn.Sequential(
            nn.Linear(vit_out_dim + embedding_dim + 1, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 1)
        )

    def forward(self, x_img, x_emb, x_actor):
        x_vit = self.vit(x_img)
        x = torch.cat([x_vit, x_emb, x_actor], dim=1)
        return self.head(x).squeeze(1)


class Regr(L.LightningModule):
    def __init__(self, model, lr=LR):
        super().__init__()
        self.model = model
        self.lr = lr
        self.mae = MeanAbsoluteError()
        self.mse = MeanSquaredError()

    def forward(self, x_img, x_emb, x_actor):
        return self.model(x_img, x_emb, x_actor)

    def _step(self, batch, tag):
        x_img, x_emb, x_actor, y = batch
        yhat = self(x_img, x_emb, x_actor)
        loss = nn.MSELoss()(yhat, y)
        self.log(f"{tag}_mae", self.mae(yhat, y), prog_bar=True, batch_size=len(y))
        self.log(f"{tag}_mse", self.mse(yhat, y), prog_bar=True, batch_size=len(y))
        return loss

    def training_step(self, b, i): return self._step(b, "train")
    def validation_step(self, b, i): return self._step(b, "val")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}


def build_loaders(df, img_size=224, bs=32, workers=2, embedding_dim=768):
    df = df[df[IMG_COL].apply(lambda p: os.path.exists(os.path.join(POSTER_DIR, p)))]
    val_df = df.sample(frac=0.1, random_state=SEED)
    tr_df  = df.drop(val_df.index)

    mean, std = [0.5]*3, [0.5]*3
    tr_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(.2, .2, .2, .1),
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    val_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(), transforms.Normalize(mean, std)])

    tr_loader = DataLoader(
        PosterDS(tr_df, POSTER_DIR, tr_tfm, embedding_dim),
        batch_size=bs, shuffle=True, num_workers=workers, pin_memory=True)
    val_loader = DataLoader(
        PosterDS(val_df, POSTER_DIR, val_tfm, embedding_dim),
        batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
    return tr_loader, val_loader


def train_one(df, embedding_dim=768, epochs=EPOCHS, bs=BATCH_SIZE, freeze_vit=False):
    tr_loader, val_loader = build_loaders(df, bs=bs, embedding_dim=embedding_dim)
    model = ViTWithTabular(embedding_dim)

    if freeze_vit:
        for param in model.vit.parameters():
            param.requires_grad = False
        print("🧊 ViT backbone is frozen.")
    else:
        print("🔥 ViT backbone is trainable.")

    lit_model = Regr(model)

    checkpoint_cb = L.pytorch.callbacks.ModelCheckpoint(
        monitor="val_mae", mode="min", save_top_k=1,
        filename="best-vit-tab-{epoch:02d}-{val_mae:.3f}"
    )

    print(f"🧪 Training on {len(tr_loader.dataset):,} samples")
    print(f"🧾 Validating on {len(val_loader.dataset):,} samples")

    trainer = L.Trainer(
        max_epochs=epochs,
        precision="16-mixed" if torch.cuda.is_available() else 32,
        accelerator="auto",
        deterministic=True,
        log_every_n_steps=10,
        callbacks=[checkpoint_cb]
    )

    t0 = time.time()
    trainer.fit(lit_model, tr_loader, val_loader)
    metrics = trainer.validate(lit_model, val_loader, verbose=False)[0]
    print(f"\n▶ VIT+TAB MAE={metrics['val_mae']:.3f}  MSE={metrics['val_mse']:.3f}  |  {(time.time() - t0)/60:.1f} min")
    print(f"📦 Best checkpoint saved to: {checkpoint_cb.best_model_path}")

    # Preview a few predictions
    lit_model.eval()
    with torch.no_grad():
        for x_img, x_emb, x_actor, y_true in val_loader:
            y_pred = lit_model(x_img.to(lit_model.device), x_emb.to(lit_model.device), x_actor.to(lit_model.device))
            print("\n🔍 Sample predictions:")
            for i in range(min(5, len(y_true))):
                print(f"🎯 True: {y_true[i].item():.2f} → 🧠 Pred: {y_pred[i].item():.2f}")
            break

    return metrics


# final_df must be preloaded DataFrame
metrics = train_one(final_df)

print("\nFinal Metrics:")
print("MAE:", metrics["val_mae"], "MSE:", metrics["val_mse"])


In [ ]:
import time
while True:
    print("Keeping session alive...")
    time.sleep(60 * 10)  # Sleep for 10 minutes

In [ ]:
import shutil

shutil.move(
    "/kaggle/working/lightning_logs/version_4/checkpoints/best-vit-tab-epoch=03-val_mae=1.668.ckpt",
    "/kaggle/working/lightning_logs/best_model.ckpt"
)

In [ ]:
import os

checkpoint_dir = "/kaggle/working/lightning_logs/version_4/checkpoints/"

if os.path.exists(checkpoint_dir):
    print("📂 Contents of checkpoints directory:")
    print(os.listdir(checkpoint_dir))
else:
    print("❌ Directory not found!")

In [ ]:
import os, random, time, gc
import pandas as pd
from PIL import Image, UnidentifiedImageError
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
import lightning as L
from torchmetrics.regression import MeanAbsoluteError, MeanSquaredError

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


POSTER_DIR = "/kaggle/input/computer-vision-project-dataset/poster_images/"
IMG_COL    = "image_path"
SCORE_COL  = "imdb_score"
EMBEDDING_DIM = 768
EPOCHS     = 10
BATCH_SIZE = 128
LR         = 1e-4


class PosterDS_Base(Dataset):
    def __init__(self, df, root, tfm):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.tfm = tfm

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.root, row[IMG_COL])

        try:
            img = Image.open(path).convert("RGB")
        except (FileNotFoundError, UnidentifiedImageError):
            return self.__getitem__((idx + 1) % len(self))

        x_img = self.tfm(img)
        y = torch.tensor(row[SCORE_COL], dtype=torch.float32)

        return x_img, y


class ResNet50Regressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet = timm.create_model("resnet50", pretrained=True)
        resnet_out_dim = self.resnet.get_classifier().in_features
        self.resnet.reset_classifier(0)

        self.head = nn.Sequential(
            nn.Linear(resnet_out_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 1)
        )

    def forward(self, x_img):
        x_resnet = self.resnet(x_img)
        return self.head(x_resnet).squeeze(1)


class Regr_Base(L.LightningModule):
    def __init__(self, model, lr=LR):
        super().__init__()
        self.model = model
        self.lr = lr
        self.mae = MeanAbsoluteError()
        self.mse = MeanSquaredError()

    def forward(self, x_img):
        return self.model(x_img)

    def _step(self, batch, tag):
        x_img, y = batch
        yhat = self(x_img)
        loss = nn.MSELoss()(yhat, y)
        self.log(f"{tag}_mae", self.mae(yhat, y), prog_bar=True, batch_size=len(y))
        self.log(f"{tag}_mse", self.mse(yhat, y), prog_bar=True, batch_size=len(y))
        return loss

    def training_step(self, b, i): return self._step(b, "train")
    def validation_step(self, b, i): return self._step(b, "val")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}


def build_loaders_base(df, img_size=224, bs=32, workers=2):
    df = df[df[IMG_COL].apply(lambda p: os.path.exists(os.path.join(POSTER_DIR, p)))]
    val_df = df.sample(frac=0.1, random_state=SEED)
    tr_df  = df.drop(val_df.index)

    mean, std = [0.5]*3, [0.5]*3
    tr_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(.2, .2, .2, .1),
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    val_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(), transforms.Normalize(mean, std)])

    tr_loader = DataLoader(
        PosterDS_Base(tr_df, POSTER_DIR, tr_tfm),
        batch_size=bs, shuffle=True, num_workers=workers, pin_memory=True)
    val_loader = DataLoader(
        PosterDS_Base(val_df, POSTER_DIR, val_tfm),
        batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
    return tr_loader, val_loader


def train_one_base(df, epochs=EPOCHS, bs=BATCH_SIZE, freeze_resnet=False):
    tr_loader, val_loader = build_loaders_base(df, bs=bs)
    model = ResNet50Regressor()

    if freeze_resnet:
        for param in model.resnet.parameters():
            param.requires_grad = False
        print("ResNet backbone is frozen.")
    else:
        print("ResNet backbone is trainable.")

    lit_model = Regr_Base(model)

    checkpoint_cb = L.pytorch.callbacks.ModelCheckpoint(
        monitor="val_mae", mode="min", save_top_k=1,
        filename="best-resnet50-{epoch:02d}-{val_mae:.3f}"
    )

    print(f"Training on {len(tr_loader.dataset)} samples")
    print(f"Validating on {len(val_loader.dataset)} samples")

    trainer = L.Trainer(
        max_epochs=epochs,
        precision="16-mixed" if torch.cuda.is_available() else 32,
        accelerator="auto",
        deterministic=True,
        log_every_n_steps=10,
        callbacks=[checkpoint_cb]
    )

    t0 = time.time()
    trainer.fit(lit_model, tr_loader, val_loader)
    metrics = trainer.validate(lit_model, val_loader, verbose=False)[0]
    print(f"\nRESNET50 MAE={metrics['val_mae']:.3f}  MSE={metrics['val_mse']:.3f}  Time={(time.time() - t0)/60:.1f} min")
    print(f"Best checkpoint saved to: {checkpoint_cb.best_model_path}")

    lit_model.eval()
    with torch.no_grad():
        for x_img, y_true in val_loader:
            y_pred = lit_model(x_img.to(lit_model.device))
            print("\nSample predictions:")
            for i in range(min(5, len(y_true))):
                print(f"True: {y_true[i].item():.2f} → Pred: {y_pred[i].item():.2f}")
            break

    return metrics


metrics_base = train_one_base(final_df)

print("\nFinal Base Model Metrics:")
print("MAE:", metrics_base["val_mae"], "MSE:", metrics_base["val_mse"])


NameError: name 'final_df' is not defined